# Simulación de Colas M/M/1
Simulación basada en eventos para estimar el número de personas atendidas en el intervalo [0, T].

In [ ]:
from random import random
from math import log, sqrt, exp, pi
from numba import jit
import matplotlib.pyplot as plt
import numpy as np

## Función de simulación
Genera `nr` simulaciones del número de personas atendidas en el intervalo [0, T].

In [ ]:
@jit(nopython=True)
def f(T, nr):
    ll = np.zeros(nr)
    for i in range(nr):
        l_l = 2          # tasa de llegadas (lambda)
        l_s = 1.8        # tasa de servicio (mu)
        t = 0            # tiempo actual
        infi = 100000000
        ts = infi        # tiempo próxima salida
        tl = 0           # tiempo próxima llegada
        l = 0            # número de personas en sistema
        cuenta = 0       # número atendido

        while min(tl, ts) < T:   # Simulación basada en eventos
            t = min(tl, ts)

            # Caso 1: llegada
            if t == tl:
                l = l + 1
                tl = t - (1 / l_l) * log(random())   # siguiente llegada
                if l == 1:
                    ts = t - (1 / l_s) * log(random())  # empieza servicio

            # Caso 2: salida
            else:
                l = l - 1
                cuenta = cuenta + 1
                if l == 0:
                    ts = infi
                else:
                    ts = t - (1. / l_s) * log(random())

        ll[i] = cuenta
    return ll

## Parámetros y ejecución de la simulación

In [ ]:
nr = 1000000   # número de repeticiones
T = 120        # tiempo total de observación

ll = f(T, nr)  # ll = [N1(120), N2(120), ..., N1000000(120)]

## Histograma y curva teórica (aproximación normal)

In [ ]:
# Construcción de bins
b = []
x = np.min(ll)
lim = np.max(ll)
while x < lim:
    b.append(x)
    x = x + 1

# Media y varianza
media = np.mean(ll)
var = np.var(ll)

# Curva teórica normal
lt = []
for x in b:
    lt.append((1 / (sqrt(2 * pi * var))) * exp(-(x - 0.5 - media) ** 2 / (2 * var)))

# Gráfica
plt.figure(figsize=(8, 4))
plt.hist(ll, density=1, bins=b, color='pink', edgecolor='black', label='Simulación')
plt.plot(b, lt, color='blue', linewidth=2, label='Aprox. Normal')
plt.xlabel('Número de personas atendidas')
plt.ylabel('Densidad')
plt.title('Distribución del número de atendidos en [0, T]')
plt.legend()
plt.tight_layout()
plt.show()

## Estadísticas: cuantiles y media

In [ ]:
q05 = np.quantile(ll, 0.05)
q95 = np.quantile(ll, 0.95)
media_final = np.mean(ll)

print(f"Cuantil 5%:  {q05}")
print(f"Cuantil 95%: {q95}")
print(f"Media:       {media_final}")